# synthetic 12 bronze handoff


## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Bronze portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify how the Bronze output becomes the stable input foundation for Silver processing.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_12_bronze_handoff_code_reference.md`


In [ ]:
import os
import pandas as pd

from typing import Any, cast

from utils.database.postgres import (
    get_engine_from_env,
    read_sql_dataframe,
)

from utils.synthetic.pipeline.bronze_handoff import (
    handoff_final_aligned_observations_to_bronze, 
    run_bronze_handoff_loop,
)

from utils.core.env_helpers import (
    env_bool, 
    env_int, 
    env_optional_int, 
    env_str,
)

---

In [ ]:
def dataframe_int_value(
    dataframe: pd.DataFrame,
    column_name: str,
    row_index: int = 0,
) -> int:
    """
    Return a dataframe scalar value as an integer.

    This helper is used in notebook validation cells where SQL aggregate
    queries return one-row dataframes. It keeps validation code readable and
    avoids Pylance warnings caused by pandas scalar typing.

    Parameters
    ----------
    dataframe:
        Dataframe containing the SQL aggregate result.
    column_name:
        Name of the column to read.
    row_index:
        Positional row index to read from. Defaults to the first row.

    Returns
    -------
    int
        Integer representation of the selected scalar value.

    Raises
    ------
    ValueError
        If the dataframe is empty, the column is missing, or the value is null.
    """
    if dataframe.empty:
        raise ValueError("Cannot read integer value from an empty dataframe.")

    if column_name not in dataframe.columns:
        raise ValueError(f"Column not found in dataframe: {column_name}")

    value = dataframe[column_name].iloc[row_index]

    if pd.isna(value):
        raise ValueError(f"Column {column_name} contains a null value at row {row_index}.")

    return int(cast(Any, value))

----

In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

BATCH_SIZE = env_int(
    "SYNTHETIC_BRONZE_HANDOFF_BATCH_SIZE",
    500,
)

# Hardcoded "replace" (not env-driven) because the Bronze input stage must
# be rebuilt from scratch each run to avoid mixing rows from different run_ids.
IF_EXISTS_FLAG = "replace"

# Bronze ingestion requires complete observations; passing incomplete rows
# would introduce nulls into Silver feature columns and invalidate model inputs.
COMPLETE_ONLY_FLAG = env_bool(
    "SYNTHETIC_BRONZE_HANDOFF_COMPLETE_ONLY",
    True,
)

STOP_ON_FAILURE_FLAG = env_bool(
    "SYNTHETIC_BRONZE_HANDOFF_STOP_ON_FAILURE",
    True,
)

MAX_ITERATIONS = env_optional_int(
    "SYNTHETIC_BRONZE_HANDOFF_MAX_ITERATIONS",
    default=None,
)

FINAL_ALIGNMENT_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_FINAL_ALIGNED_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_final_aligned_stage",
    aliases=("FINAL_ALIGNMENT_SOURCE_TABLE_NAME",),
)

BRONZE_HANDOFF_TARGET_TABLE_NAME = env_str(
    "BRONZE_HANDOFF_OBSERVATIONS_TABLE",
    "bronze_observations_input_stage",
    aliases=("BRONZE_HANDOFF_TARGET_TABLE_NAME",),
)

print("Bronze handoff config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("source table:", FINAL_ALIGNMENT_SOURCE_TABLE_NAME)
print("target table:", BRONZE_HANDOFF_TARGET_TABLE_NAME)
print("batch size:", BATCH_SIZE)
print("max iterations:", MAX_ITERATIONS)

In [ ]:
# RUN_HANDOFF guards the actual write; set False to inspect preflight results
# without committing rows to the Bronze target table.
RUN_HANDOFF = True

RUN_MODE = env_str(
    "SYNTHETIC_BRONZE_HANDOFF_RUN_MODE",
    "full_batch",
    aliases=("BRONZE_HANDOFF_RUN_MODE",),
)

# Validate mode before the handoff loop to surface config errors before
# any rows are written (avoids partial writes on bad config).
if RUN_MODE not in {"row", "row_batch", "full_batch"}:
    raise ValueError(
        "RUN_MODE must be one of: 'row', 'row_batch', 'full_batch'. "
        f"Received: {RUN_MODE!r}"
    )

In [ ]:
# Local override
# Script style
# Mode
# "row" | "row_batch" | "full_batch"

#RUN_MODE = "row_batch"


In [ ]:
print(f"Bronze handoff run mode: {RUN_MODE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Complete only: {COMPLETE_ONLY_FLAG}")
print(f"Source table: {SCHEMA}.{FINAL_ALIGNMENT_SOURCE_TABLE_NAME}")
print(f"Target table: {SCHEMA}.{BRONZE_HANDOFF_TARGET_TABLE_NAME}")

---

In [ ]:
engine = get_engine_from_env()

----


### Preflight validation for Stage 12 source table


In [ ]:


# Preflight reads the source row count before the handoff loop so we can
# fail fast if Stage 11 output is missing, rather than completing a no-op run.
source_preflight_df = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS source_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_row_count,
        COUNT(*) FILTER (WHERE bronze_handoff_status IS NULL) AS null_handoff_status_count,
        COUNT(*) FILTER (WHERE bronze_handoff_status = 'pending') AS pending_count,
        COUNT(*) FILTER (WHERE bronze_handoff_status = 'completed') AS completed_count,
        COUNT(*) FILTER (WHERE bronze_handoff_status = 'failed') AS failed_count
    FROM "{SCHEMA}"."{FINAL_ALIGNMENT_SOURCE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(source_preflight_df)

source_row_count = dataframe_int_value(source_preflight_df, "source_row_count")
complete_row_count = dataframe_int_value(source_preflight_df, "complete_row_count")

if source_row_count == 0:
    raise ValueError(
        "Stage 12 source table has zero rows for the active dataset/run. "
        "Run Stage 11 final alignment before Stage 12."
    )

if COMPLETE_ONLY_FLAG and complete_row_count == 0:
    raise ValueError(
        "Stage 12 is configured for complete_only=True, but no rows have "
        "rebuild_is_complete=True. Check Stage 11 output."
    )

print("Stage 12 source preflight passed.")

----

### Run Bronze handoff


In [ ]:


if RUN_HANDOFF:
    results = run_bronze_handoff_loop(
        engine=engine,
        mode=RUN_MODE,
        batch_size=BATCH_SIZE,
        schema=SCHEMA,
        source_table=FINAL_ALIGNMENT_SOURCE_TABLE_NAME,
        target_table=BRONZE_HANDOFF_TARGET_TABLE_NAME,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        complete_only=COMPLETE_ONLY_FLAG,
        max_iterations=MAX_ITERATIONS,
        stop_on_failure=STOP_ON_FAILURE_FLAG,
    )
else:
    results = []

latest_result = results[-1] if results else {"status": "not_run"}

print("Bronze handoff run mode:", RUN_MODE)
print("Bronze handoff latest result:")
print(latest_result)

display(pd.DataFrame(results))

---

### Post-handoff validation


In [ ]:


target_validation_df = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS target_row_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_index_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        COUNT(*) FILTER (WHERE observation_timestamp IS NULL) AS null_timestamp_count,
        COUNT(*) FILTER (WHERE machine_status IS NULL) AS null_machine_status_count
    FROM "{SCHEMA}"."{BRONZE_HANDOFF_TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

source_status_df = read_sql_dataframe(
    engine,
    f"""
    SELECT
        bronze_handoff_status,
        COUNT(*) AS row_count
    FROM "{SCHEMA}"."{FINAL_ALIGNMENT_SOURCE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY bronze_handoff_status
    ORDER BY bronze_handoff_status
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(target_validation_df)
display(source_status_df)

target_row_count = dataframe_int_value(target_validation_df, "target_row_count")
null_timestamp_count = dataframe_int_value(target_validation_df, "null_timestamp_count")
null_machine_status_count = dataframe_int_value(target_validation_df, "null_machine_status_count")

if target_row_count == 0:
    raise ValueError("Bronze handoff target table has zero rows after Stage 12.")

# Null timestamps would silently propagate into Silver time-based features;
# hard fail here so the issue is caught at the Bronze boundary.
if null_timestamp_count > 0:
    raise ValueError(
        f"Bronze handoff target has {null_timestamp_count:,} null observation timestamps."
    )

# Null machine_status would break the label column used by all Gold models;
# raise here rather than propagating the defect into Silver.
if null_machine_status_count > 0:
    raise ValueError(
        f"Bronze handoff target has {null_machine_status_count:,} null machine_status values."
    )

print("Stage 12 Bronze handoff validation passed.")

In [ ]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Bronze step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

The Bronze handoff prepares synthetic observations for the Bronze medallion layer.
